# xLAM Llama 3.1 qLoRA Training on Google Colab

This notebook trains a function-calling model using qLoRA on Google Colab.

## Requirements:
- **Colab Pro+ recommended** (for A100 40GB GPU)
- **Colab Pro** might work (if you get A100)
- **Colab Free** won't work well (T4 has only 16GB)

## Before Starting:
1. Accept licenses:
   - Llama 3.1: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
   - xLAM: https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k
2. Get HuggingFace token: https://huggingface.co/settings/tokens
3. Get W&B API key: https://wandb.ai/authorize

## Step 1: Check GPU Allocation

In [ ]:
# Check what GPU you got
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if vram_gb >= 38:  # A100 40GB
        print("\n✅ Perfect! You got an A100. This will work great!")
    elif vram_gb >= 22:  # 24GB cards
        print("\n⚠️ You got a 24GB GPU. This might work with reduced batch size.")
        print("We'll modify the config to fit.")
    else:
        print("\n❌ GPU has insufficient VRAM for this project.")
        print("Try: Runtime -> Change runtime type -> Select 'A100 GPU'")
        print("Or consider Colab Pro+ for better GPU allocation.")

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create a directory for our project in Google Drive
!mkdir -p /content/drive/MyDrive/xlam-llama-qlora

# Set working directory
import os
os.chdir('/content')
print("Current directory:", os.getcwd())

## Step 3: Upload Project Files

**Option A**: Upload the entire `xlam-llama-qlora` folder to your Google Drive at `MyDrive/xlam-llama-qlora/`

**Option B**: Clone from GitHub (if you pushed it)

In [ ]:
# Option A: Copy from Google Drive (if you uploaded the folder)
!cp -r /content/drive/MyDrive/xlam-llama-qlora /content/

# Option B: Clone from GitHub (uncomment if using)
# !git clone https://github.com/yourusername/xlam-llama-qlora.git

# Navigate to project
os.chdir('/content/xlam-llama-qlora')
!ls -la

## Step 4: Adjust Config for GPU (if needed)

In [ ]:
import torch

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

# Read current config
with open('configs/config.yaml', 'r') as f:
    config = f.read()

# IMPORTANT: Set output directory to Google Drive to persist checkpoints
config = config.replace(
    'output_dir: "outputs/llama31-8b-xlam-lora"',
    'output_dir: "/content/drive/MyDrive/xlam-llama-qlora/outputs/llama31-8b-xlam-lora"'
)

if vram_gb < 30:  # Not an A100
    print(f"Detected {vram_gb:.1f}GB VRAM - Adjusting config for smaller GPU...")
    
    # Modify for smaller GPU
    config = config.replace('per_device_train_batch_size: 4', 'per_device_train_batch_size: 1')
    config = config.replace('gradient_accumulation_steps: 4', 'gradient_accumulation_steps: 16')
    config = config.replace('max_seq_length: 2048', 'max_seq_length: 1024')
    config = config.replace('num_train_epochs: 3', 'num_train_epochs: 1')
    
    print("✅ Config adjusted:")
    print("  - Batch size: 1 (from 4)")
    print("  - Gradient accumulation: 16 (from 4)")
    print("  - Sequence length: 1024 (from 2048)")
    print("  - Epochs: 1 (from 3) - for faster testing")
else:
    print("✅ A100 detected - using default config (optimal settings)")

# Write modified config
with open('configs/config.yaml', 'w') as f:
    f.write(config)

print("\n✅ Output directory set to Google Drive")
print("   Checkpoints will persist across Colab restarts!")

## Step 5: Install Dependencies (Fast - Core Only)

**Fast installation** - Only core training dependencies (2-3 minutes)

**Skipped:** `lm-eval` (not needed for training or xLAM evaluation, only for MMLU benchmarks)

In [ ]:
# Install PyTorch with CUDA support
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install core training dependencies (fast - 2-3 min)
!pip install -q transformers trl peft bitsandbytes accelerate datasets pyyaml sentencepiece scikit-learn wandb tqdm matplotlib

# Fix fsspec version conflict
!pip install -q fsspec==2025.3.0 --upgrade

# Verify installation
import torch
import transformers
import trl
import peft
import bitsandbytes

print("\n✅ Core packages installed:")
print(f"  - torch: {torch.__version__}")
print(f"  - transformers: {transformers.__version__}")
print(f"  - trl: {trl.__version__}")
print(f"  - peft: {peft.__version__}")
print(f"  - bitsandbytes: {bitsandbytes.__version__}")
print(f"  - CUDA available: {torch.cuda.is_available()}")

print("\n💡 Note: lm-eval skipped for faster setup. Install later if you need MMLU benchmarks.")

# Try to install Flash Attention 2 (optional, for speed)
try:
    !pip install -q flash-attn --no-build-isolation
    print("\n✅ Flash Attention 2 installed - training will be 2-3x faster!")
except:
    print("\n⚠️ Flash Attention 2 failed to install - using standard attention (slower but works)")

## Step 6: Authentication

### HuggingFace Login

In [ ]:
from huggingface_hub import login

# Enter your HuggingFace token when prompted
# Get it from: https://huggingface.co/settings/tokens
login()

### Weights & Biases Login

In [ ]:
import wandb

# Enter your W&B API key when prompted
# Get it from: https://wandb.ai/authorize
wandb.login()

## Step 7: Prepare Data (Optimized)

Using the optimized data prep script with quality filtering and caching.

In [ ]:
# Download and format xLAM dataset (optimized version)
!python -m src.data_prep_optimized

# This will:
# - Download 60k examples from HuggingFace
# - Validate and filter low-quality examples
# - Format with Llama 3.1 chat template
# - Cache formatted data for faster reruns
# - Filter by length based on max_seq_length
# - Create train/val/test splits
# - Export dataset statistics
# Takes ~5-10 minutes first run, ~10 seconds on subsequent runs (cached)

## Step 7b: Analyze Dataset (Optional)

Inspect dataset statistics and distributions.

In [ ]:
# Analyze dataset (optional - provides insights into data distribution)
!python -m src.utils.analyze_data

# This will show:
# - Text length distributions
# - Tool usage statistics
# - Answer count distribution
# - Plots saved to outputs/

In [ ]:
## Step 8: Start Training (with Optimizations)

⚠️ **Important**: Training will take 8-12 hours on A100 (or 1-2 hours if you reduced to 1 epoch)

💡 **Tip**: Keep this tab open or the runtime will disconnect after ~90 minutes of inactivity

**Optimizations enabled:**
- Flash Attention 2 (if available)
- Gradient checkpointing
- Mixed precision (bf16)
- 4-bit quantization

## Step 8: Start Training

⚠️ **Important**: Training will take 4-6 hours on A100 with Flash Attention (8-12 hours without)

💡 **Tip**: Keep this tab open or the runtime will disconnect after ~90 minutes of inactivity

**Optimizations enabled:**
- Flash Attention 2 (if available) - 2-3x faster
- Gradient checkpointing
- Mixed precision (bf16)
- 4-bit quantization
- Paged AdamW optimizer

In [ ]:
# Start training
!python -m src.train

# W&B will print a URL like: https://wandb.ai/yourname/xlam-llama-qlora/runs/xxx
# Open it to monitor training progress in real-time!

## Step 8b: Merge Adapter (Optional but Recommended)

Merge LoRA weights into base model for 20-30% faster inference.

**Note**: This is optional but recommended for deployment and faster evaluation.

In [ ]:
# Merge LoRA adapter into base model
!python -m src.merge_adapter

# This creates a merged model at outputs/merged_model/
# Benefits:
# - No PEFT dependency at inference time
# - 20-30% faster inference
# - Simpler deployment

print("\n✅ Merged model saved to outputs/merged_model/")
print("You can now use this for faster inference and evaluation!")

## Step 9: Evaluate Model

In [ ]:
# After training completes, run custom xLAM evaluation
!python -m src.eval

# This compares base model vs fine-tuned model on tool calling tasks
# Results saved to outputs/eval_results.json

## Step 10: View Results

In [ ]:
import json

# Load and display evaluation results
with open('outputs/eval_results.json', 'r') as f:
    results = json.load(f)

print("=" * 80)
print("EVALUATION RESULTS")
print("=" * 80)
print(json.dumps(results, indent=2))

# Print comparison table
base = results['base_model']['metrics']
lora = results['lora_model']['metrics']

print("\n" + "=" * 80)
print(f"{'Metric':<30} {'Base Model':<15} {'LoRA Model':<15} {'Improvement':<15}")
print("-" * 80)
for metric in ['json_validity_rate', 'tool_name_accuracy', 'arg_key_accuracy', 'arg_value_exact_accuracy']:
    b = base[metric]
    l = lora[metric]
    imp = l - b
    print(f"{metric:<30} {b:>14.2%} {l:>14.2%} {imp:>+14.2%}")

## Step 10b: LM Evaluation Harness (MMLU) - Optional

Benchmark on standard academic tasks using EleutherAI's evaluation harness.

**Note**: This requires installing `lm-eval` first (15-20 min install time). Skip if you only need xLAM evaluation.

In [ ]:
# Install lm-eval first (takes 15-20 minutes)
!pip install lm-eval>=0.4.0

# Run LM Evaluation Harness (MMLU benchmark)
!python -m src.eval_lm_harness

# This evaluates both base and LoRA models on MMLU
# Takes 30-60 minutes depending on GPU
# Results saved to outputs/lm_harness_results/

In [ ]:
# View MMLU results
import json
from pathlib import Path

lm_results_dir = Path('outputs/lm_harness_results')

if (lm_results_dir / 'combined_results.json').exists():
    with open(lm_results_dir / 'combined_results.json', 'r') as f:
        lm_results = json.load(f)
    
    print("=" * 80)
    print("LM HARNESS RESULTS (MMLU)")
    print("=" * 80)
    
    # Check if results contain actual MMLU scores
    if 'base_model_results' in lm_results:
        print("\nBase Model:")
        print(json.dumps(lm_results['base_model_results'], indent=2)[:500] + "...")
        
        print("\nLoRA Model:")
        print(json.dumps(lm_results['lora_model_results'], indent=2)[:500] + "...")
else:
    print("LM Harness results not found. Run the evaluation first.")

## Step 10c: Frontier Model Comparison (Optional)

Compare your model against GPT-4, Claude, and Gemini on the same test set.

**Requirements**:
- API keys for the models you want to test
- Budget for API calls (~$0.30-1.00 per 100 examples)

**Note**: This is optional but provides valuable benchmarking data.

In [ ]:
# Setup API keys for frontier models (optional)
# Only set the keys for models you want to test

import os
from getpass import getpass

# Uncomment and enter API keys for models you want to test:

# OpenAI (GPT-4, GPT-4o)
# os.environ['OPENAI_API_KEY'] = getpass('Enter OpenAI API key: ')

# Anthropic (Claude)
# os.environ['ANTHROPIC_API_KEY'] = getpass('Enter Anthropic API key: ')

# Google (Gemini)
# os.environ['GOOGLE_API_KEY'] = getpass('Enter Google API key: ')

print("API keys configured. Edit configs/config.yaml to enable specific models.")

In [ ]:
# Install API clients first (only needed for frontier comparison)
!pip install -q openai anthropic google-generativeai

# Run frontier model comparison (if API keys configured)
# This will only run for models that are:
# 1. Enabled in configs/config.yaml
# 2. Have API keys set

!python -m src.eval_frontier

# Results saved to outputs/frontier_comparison/

In [ ]:
# View frontier model comparison results
import json
from pathlib import Path

frontier_results_path = Path('outputs/frontier_comparison/comparison_results.json')

if frontier_results_path.exists():
    with open(frontier_results_path, 'r') as f:
        frontier_results = json.load(f)
    
    print("=" * 100)
    print("FRONTIER MODEL COMPARISON")
    print("=" * 100)
    
    # Print comparison table
    results = frontier_results['results']
    model_names = list(results.keys())
    
    print(f"\n{'Model':<25}", end="")
    print(f"{'JSON Validity':>15}{'Tool Name Acc':>15}{'Arg Key Acc':>15}{'Arg Val Exact':>15}")
    print("-" * 100)
    
    for model_name in model_names:
        metrics = results[model_name]['metrics']
        print(f"{model_name:<25}", end="")
        print(f"{metrics['json_validity_rate']:>14.2%}", end="")
        print(f"{metrics['tool_name_accuracy']:>15.2%}", end="")
        print(f"{metrics['arg_key_accuracy']:>15.2%}", end="")
        print(f"{metrics['arg_value_exact_accuracy']:>15.2%}")
    
    print("\n💡 Tip: Your specialized 8B model may outperform larger general-purpose models on tool calling!")
else:
    print("Frontier comparison results not found. Configure API keys and run the evaluation first.")

## Bonus: Performance Optimizations

The project includes several optimization utilities. Here are some examples:

In [ ]:
# Example: Using batched inference for faster evaluation
from src.utils.batched_inference import generate_batch

# This is 3-4x faster than processing one at a time!
# Uncomment to test:

# from transformers import AutoModelForCausalLM, AutoTokenizer
# 
# model = AutoModelForCausalLM.from_pretrained(
#     "outputs/merged_model",
#     device_map="auto",
# )
# tokenizer = AutoTokenizer.from_pretrained("outputs/merged_model")
# 
# test_prompts = [
#     "What's the weather in Tokyo?",
#     "Book a flight to Paris",
#     "Set a reminder for tomorrow",
# ]
# 
# responses = generate_batch(
#     model=model,
#     tokenizer=tokenizer,
#     prompts=test_prompts,
#     batch_size=4,
#     max_new_tokens=512,
# )
# 
# for prompt, response in zip(test_prompts, responses):
#     print(f"Prompt: {prompt}")
#     print(f"Response: {response}")
#     print("-" * 80)

print("See OPTIMIZATIONS.md for more optimization examples!")

### Optimization Resources

For detailed information on all optimizations:

1. **OPTIMIZATIONS.md** - Complete optimization guide with benchmarks
2. **src/utils/** - Utility modules for:
   - `data_quality.py` - Data validation and filtering
   - `training_utils.py` - Training callbacks and tools
   - `batched_inference.py` - Fast batched generation
   - `analyze_data.py` - Dataset analysis tools

**Quick wins:**
- ✅ Optimized data prep (30-60x faster on reruns)
- ✅ Flash Attention 2 (2-3x faster training)
- ✅ Merged adapter (1.3x faster inference)
- ✅ Batched inference (3-4x faster evaluation)

## Step 11: Test Inference

In [ ]:
# Test the model with a sample query
!python -m src.inference --query "get the weather in Tokyo"

## Step 12: Save Results to Google Drive

In [ ]:
# Copy outputs to Google Drive for safekeeping
!cp -r outputs /content/drive/MyDrive/xlam-llama-qlora/
!cp -r data /content/drive/MyDrive/xlam-llama-qlora/

print("✅ Results saved to Google Drive!")
print("\nSaved:")
print("  - LoRA adapter: /content/drive/MyDrive/xlam-llama-qlora/outputs/llama31-8b-xlam-lora/")
print("  - Merged model: /content/drive/MyDrive/xlam-llama-qlora/outputs/merged_model/")
print("  - Custom evaluation: /content/drive/MyDrive/xlam-llama-qlora/outputs/eval_results.json")
print("  - LM Harness (MMLU): /content/drive/MyDrive/xlam-llama-qlora/outputs/lm_harness_results/")
print("  - Frontier comparison: /content/drive/MyDrive/xlam-llama-qlora/outputs/frontier_comparison/")
print("  - Data splits: /content/drive/MyDrive/xlam-llama-qlora/data/")
print("  - Dataset statistics: /content/drive/MyDrive/xlam-llama-qlora/data/dataset_statistics.json")

## Tips for Long Training Sessions

### Prevent Disconnection:
1. **Keep tab active** - Colab disconnects after ~90 min of inactivity
2. **Use Colab Pro/Pro+** - Longer timeouts (up to 24 hours)
3. **Run this code** to keep connection alive:

```javascript
// Open browser console (F12) and paste:
function ClickConnect(){
  console.log("Keeping alive...");
  document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)
```

### Monitor Progress:
- Watch W&B dashboard (works even if you close laptop)
- Check GPU usage in Colab: Runtime → Manage sessions

### If Disconnected:
- Reconnect and run: `!ls outputs/llama31-8b-xlam-lora/checkpoint-*`
- Training resumes from last checkpoint automatically